Write a solution to report the fraction of players that logged in again on the day after the day they first logged in, rounded to 2 decimal places. In other words, you need to determine the number of players who logged in on the day immediately following their initial login, and divide it by the number of total players.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [0]:
#️⃣ creating or retrieving spark session
spark = SparkSession.builder.appName("SQL50_21").getOrCreate()

#️⃣ creating list of tuples for Activity data
activity_data = [(1, 2, "2016-03-01", 5),(1, 2, "2016-03-02", 6),
                          (2, 3, "2017-06-25", 1),(3, 1, "2016-03-02", 0),
                          (3, 4, "2018-07-03", 5)]

#️⃣ specifying columns
activity_cols = ("player_id","device_id","event_date","games_played")

#️⃣ creating dataframe
activity_df = spark.createDataFrame(activity_data, activity_cols)

In [0]:
display(activity_df)

In [0]:
from pyspark.sql.window import Window

# Find each player's first login date
first_login_df = activity_df.groupBy("player_id").agg(F.min("event_date").alias("first_login_date"))

# Find if player logged in the day after their first login
activity_with_first = activity_df.join(first_login_df, "player_id",how='inner')
next_day_df = activity_with_first.withColumn(
    "is_next_day",
    F.when(
        F.datediff("event_date", "first_login_date") == 1, 1
    ).otherwise(0)
)

# For each player, check if they logged in the day after their first login
player_next_day = next_day_df.groupBy("player_id").agg(F.max("is_next_day").alias("logged_next_day"))

# Calculate the fraction
fraction = player_next_day.agg(
    F.round(F.sum("logged_next_day") / F.count("player_id"), 2).alias("fraction")
)

display(fraction)